# Building agent loop: Client SDK

Tool definition

In [1]:
import json

TOOLS = [
    {
        "name": "get_order_status",
        "description": (
            "Get the status of an order by providing the order ID."
            "Use this when customers ask about their order, shipment or delivery."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The unique identifier of the order.",
                }
            },
            "required": ["order_id"],
        },
    }
]

def get_order_status(order_id: str) -> str: 
    mock_orders = {
        "5678": "Your order #5678 is currently being processed and will be shipped soon.",
        "1234": "Your order #1234 has been shipped and is expected to arrive"
    }
    return mock_orders.get(order_id, f"Order ID {order_id} not found. Please check the order ID and try again.")

Writing the agent loop

In [ ]:
import anthropic
import os
from dotenv import load_dotenv
load_dotenv()

model = "claude-opus-4-8"
client = anthropic.Anthropic()

SYSTEM_PROMPT = "You are a customer support agent. Help customers with order enqueries."

def run_agent(task: str) -> str:
    messages = [{"role": "user", "content": task}]

    while True:
        response = client.messages.create(
            model = model,
            max_tokens=1000,
            system = SYSTEM_PROMPT,
            tools = TOOLS,
            messages = messages
        )
        print(response.stop_reason)
        if response.stop_reason == "end_turn":
            for block in response.content:
                if block.type == "text":
                    return block.text

        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = dispatch_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            messages.append({"role": "user", "content": tool_results})

def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    if tool_name == "get_order_status":
        order_id = tool_input.get("order_id")
        return get_order_status(order_id)
    else:
        return f"Tool {tool_name} not recognized."

Running the agent

In [9]:
answer = run_agent("Can you tell me the status of my order with ID 5678?")
print("================================")
print("Agent's response:", answer)

Message(id='msg_011CgU78StLHzst7cj6Xu2qH', container=None, content=[TextBlock(citations=None, text="I'll check the status of your order right away.", type='text'), ToolUseBlock(id='toolu_01PRJ2nhwfsojy7AR5Qtxnru', caller=DirectCaller(type='direct'), input={'order_id': '5678'}, name='get_order_status', type='tool_use', toolset_name=None)], diagnostics=None, model='claude-opus-4-8', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=452, output_tokens=65, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard'))
Message(id='msg_011CgU78YXuCz6nAPLYideP9', container=None, content=[TextBlock(citations=None, text="Here's the latest on your order:\n\n**Order #5678** is currently being **